# MDTest · Python Tutorial

This **optional tutorial** demonstrates MDTest through Python, with worked examples and figures.
For routine analysis without programming, start with **`MDTestWorkbench.ipynb`** in the repository root.

**How to open:** use the MDTest START launcher to open JupyterLab, then open
`tutorials/MDTest_Tutorial.ipynb` in its file browser and select **Run → Run All Cells**.
The startup cell automatically finds the example data in the repository root.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import Image, Markdown, display

# Jupyter normally starts this notebook inside tutorials/. Find the
# repository root so examples/data works without manual path edits.
HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / "pyproject.toml").is_file() and (p / "examples" / "data").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open the tutorial from inside the extracted MDTest repository using the START launcher.")
os.chdir(ROOT)

# The START launcher already installs mdtest. Also support source checkout
# users who have not installed it yet. This affects only the tutorial.
if (ROOT / "src" / "mdtest").is_dir() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import mdtest

OUT = ROOT / "notebook_results"
OUT.mkdir(exist_ok=True)

def show_text(path):
    path = Path(path)
    display(Markdown(f"```text\n{path.read_text()}\n```"))

def show_image(path):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Missing image: {path}")

print(f"MDTest {mdtest.__version__} · example data: {ROOT / 'examples' / 'data'}")


## 1. Manual NarK volume example

This reproduces a simple manual run on the NarK volume time series after discarding the initial transient and using block size `m=30`. The command writes a text report, a JSON report, and two PNG diagnostic figures.

In [ ]:
manual_dir = OUT / "manual_nark"
manual_dir.mkdir(exist_ok=True)
manual = mdtest.manual(
    "examples/data/NarK_TimeSeriesData.csv",
    delim="comma", drop_nan=True, col=6, ts=600000, m=30, n="max",
    outdir=manual_dir, prefix="nark_volume_manual",
    write_outputs=True, make_plots=True,
)
print(manual)
show_text(manual_dir / "nark_volume_manual.txt")
show_image(manual_dir / "nark_volume_manual_raw.png")
show_image(manual_dir / "nark_volume_manual_blocks.png")


## 2. Automatic stationarity/correlation search on a synthetic stress test

The variance-relaxation data set is a controlled synthetic benchmark. The mean is zero by construction, but the fluctuation width relaxes. This example demonstrates the `auto-safe` workflow.

In [ ]:
var_dir = OUT / "auto_safe_variance_relaxation"
var_dir.mkdir(exist_ok=True)
var_result = mdtest.auto_safe(
    "examples/data/synthetic_variance_relaxation_independent.csv",
    delim="comma", col=1,
    m_trend_grid=[2, 5, 10, 20, 40],
    m_corr_grid=[10, 20, 40, 80, 160, 320],
    outdir=var_dir, prefix="var_relax",
    write_outputs=True, make_plots=True,
)
print({
    "status": var_result.status,
    "selected_ts_SW": var_result.selected_ts_sw,
    "selected_start_index_SW": var_result.selected_start_index_sw,
    "selected_m_corr": var_result.selected_m_corr,
    "selected_n_corr": var_result.selected_n_corr,
})
selected_m = var_result.selected_m_corr
show_text(var_dir / f"var_relax_m{selected_m}.txt")
show_image(var_dir / f"var_relax_m{selected_m}_raw.png")
show_image(var_dir / f"var_relax_m{selected_m}_blocks.png")


## 3. ESS/IACT comparator

The `auto-compare` mode runs the MDTest stationarity screen and a Chodera/PyMBAR-style ESS/IACT start-time comparator, then reports a conservative start as the maximum of the two candidate starts.

In [ ]:
cmp_dir = OUT / "auto_compare_mean_drift"
cmp_dir.mkdir(exist_ok=True)
cmp_result = mdtest.auto_compare(
    "examples/data/synthetic_mean_drift_independent.csv",
    delim="comma", col=1,
    m_trend_grid=[2, 5, 10, 20, 40],
    m_corr_grid=[10, 20, 40, 80, 160, 320],
    outdir=cmp_dir, prefix="mean_drift",
    write_outputs=True, make_plots=True,
)
print({
    "status": cmp_result.status,
    "start_SW": cmp_result.auto_safe.selected_ts_sw,
    "start_ESS": cmp_result.ess.selected_ts_ess,
    "selected_ts_star": cmp_result.selected_ts_star,
    "selected_m_corr_star": cmp_result.selected_m_corr_star,
})
selected_m = cmp_result.selected_m_corr_star
show_text(cmp_dir / f"mean_drift_star_m{selected_m}.txt")
show_image(cmp_dir / f"mean_drift_star_m{selected_m}_raw.png")
show_image(cmp_dir / f"mean_drift_star_m{selected_m}_blocks.png")


## 4. Other interfaces

The same engine is also available as a command-line application:

```bash
mdtest auto-safe --file trajectory.csv --col 1
```

or, from an uninstalled source checkout, through the backward-compatible launcher:

```bash
python MDTest.py auto-safe --file trajectory.csv --col 1
```

Running `python MDTest.py` with no arguments starts the guided terminal walkthrough.
